# Augmento Fusion V1 — BTC Trading Strategy Backtest

This notebook backtests the **Fusion V1** market indicator from [Augmento](https://augmento.ai), a sentiment-based trading signal for Bitcoin.

## Strategy

- **LONG BTC** when the indicator is `bullish` (value = 1)
- **CASH** when the indicator is `neutral` (value = 0)

## Signal Timing

The market indicator timestamped day T is computed from sentiment data of day T. It becomes available via the API by ~00:25 UTC on T+1.

| Step | Time (UTC) | What happens |
|------|-----------|--------------|
| Signal available | T+1, 00:25 | Market indicator for day T available via API |
| **Execution** | **T+1, 01:00** | **Enter/exit at open of 01:00 UTC candle** |
| Next execution | T+2, 01:00 | Position adjusted based on day T+1's signal |

**Returns** are measured between consecutive 01:00 UTC execution prices: `exec_price[T+2] / exec_price[T+1] - 1`.

## Data Sources

| Source | Endpoint | Auth |
|--------|----------|------|
| Augmento Market Indicator | `GET https://augmento.ai/api/v1/market-indicator/1` | Free (public) or API key for real-time |
| Binance BTC/USDT Klines | `GET https://api.binance.com/api/v3/klines` | None (public) |

## Output

A full **quantstats** tear sheet with 30+ KPIs, equity curves, drawdown charts, monthly heatmaps, and more.

In [ ]:
%pip install quantstats requests pandas -q

In [ ]:
import requests
import pandas as pd
import numpy as np
import quantstats as qs
from datetime import datetime, timezone
from pathlib import Path

qs.extend_pandas()

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

AUGMENTO_API_KEY = "free"  # Enter your API key, or "free" for public access (delayed data)
USE_CACHED_DATA = True     # If True, load cached signal+price CSV when available (faster re-runs)
START_DATE = "2018-01-01"  # Backtest start date (2017 has incomplete data)

AUGMENTO_BASE_URL = "https://augmento.ai/api/v1"
MARKET_INDICATOR_ID = 1  # Fusion V1 (Bitcoin)
BINANCE_SYMBOL = "BTCUSDT"
CACHE_FILE = Path("fusion_v1_data.csv")

## 1. Fetch Market Indicator Signal from Augmento API

The `/market-indicator/{id}` endpoint returns daily signals with:
- `market_indicator`: numeric value (1 = bullish, 0 = neutral, -1 = bearish)
- `market_indicator_human`: human-readable label
- `datetime`: ISO 8601 timestamp

No signal mapping needed — the API provides the numeric position directly.

In [ ]:
def to_naive_date(dt_series: pd.Series) -> pd.Series:
    """Convert a datetime series to timezone-naive dates."""
    if dt_series.dt.tz is not None:
        return dt_series.dt.tz_convert("UTC").dt.tz_localize(None).dt.normalize()
    return dt_series.dt.normalize()


start_cutoff = pd.Timestamp(START_DATE)

if USE_CACHED_DATA and CACHE_FILE.exists():
    print(f"Loading cached data from {CACHE_FILE}")
    cached = pd.read_csv(CACHE_FILE, parse_dates=["datetime"])
    cached["date"] = to_naive_date(cached["datetime"])
    cached = cached[cached["date"] >= start_cutoff].reset_index(drop=True)
    signal_df = cached[["date", "datetime", "position", "market_indicator_human"]].copy()
    price_df = cached[["date", "open", "high", "low", "close", "volume", "exec_price"]].copy()
    print(f"  {len(cached)} rows, {cached['date'].min().date()} to {cached['date'].max().date()}")
    print(f"\nSignal distribution:")
    print(signal_df["market_indicator_human"].value_counts().to_string())
else:
    headers = {}
    if AUGMENTO_API_KEY != "free":
        headers["Authorization"] = f"Bearer {AUGMENTO_API_KEY}"

    url = f"{AUGMENTO_BASE_URL}/market-indicator/{MARKET_INDICATOR_ID}"
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    payload = resp.json()

    print(f"Indicator:       {payload['name']}")
    print(f"Reference asset: {payload['reference_asset_name']} ({payload['reference_asset_symbol']})")
    print(f"Lag days:        {payload['lag_days']}")
    print(f"Data points:     {len(payload['data'])}")

    # Build DataFrame — keep exact datetime from API
    signal_df = pd.DataFrame(payload["data"])
    signal_df["datetime"] = pd.to_datetime(signal_df["datetime"])
    signal_df["date"] = to_naive_date(signal_df["datetime"])
    signal_df = signal_df.rename(columns={"market_indicator": "position"})[["date", "datetime", "position", "market_indicator_human"]]
    signal_df = signal_df.sort_values("date").drop_duplicates(subset="date", keep="last").reset_index(drop=True)

    # Apply start date filter
    signal_df = signal_df[signal_df["date"] >= start_cutoff].reset_index(drop=True)

    print(f"\nDate range: {signal_df['date'].min().date()} to {signal_df['date'].max().date()}")
    print(f"\nSignal distribution:")
    print(signal_df["market_indicator_human"].value_counts().to_string())

signal_df.tail()

## 2. Fetch BTC/USDT Hourly Klines from Binance

Binance's public API returns up to 1000 klines per request. We paginate from the earliest signal date to collect full hourly history. We keep the hourly granularity to extract the **01:00 UTC execution price** for each day.

In [ ]:
def fetch_binance_klines(symbol: str, interval: str, start_date: datetime) -> pd.DataFrame:
    """Fetch all klines from Binance by paginating in batches of 1000."""
    url = "https://api.binance.com/api/v3/klines"
    start_ms = int(start_date.timestamp() * 1000)
    all_klines = []

    while True:
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": start_ms,
            "limit": 1000,
        }
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        batch = resp.json()

        if not batch:
            break

        all_klines.extend(batch)
        # Next batch starts after the last candle's open time
        start_ms = batch[-1][0] + 1

        if len(batch) < 1000:
            break

    print(f"Fetched {len(all_klines):,} hourly klines")

    df = pd.DataFrame(all_klines, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "trades", "taker_buy_base",
        "taker_buy_quote", "ignore",
    ])
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)
    df = df[["open_time", "open", "high", "low", "close", "volume"]].copy()
    return df


if not (USE_CACHED_DATA and CACHE_FILE.exists()):
    # Fetch hourly data starting from the first signal date
    start = datetime(
        signal_df["date"].min().year,
        signal_df["date"].min().month,
        signal_df["date"].min().day,
        tzinfo=timezone.utc,
    )
    klines = fetch_binance_klines(BINANCE_SYMBOL, "1h", start)

    # Extract the 01:00 UTC candle for each day — this is the execution candle.
    # Signal for day T becomes available ~00:25 UTC on T+1.
    # We execute at 01:00 UTC on T+1 (open of the 01:00 candle).
    klines_01 = klines[klines["open_time"].dt.hour == 1].copy()
    klines_01["date"] = to_naive_date(klines_01["open_time"])

    # Daily OHLCV from all hourly candles (for context)
    daily_ohlcv = klines.copy()
    daily_ohlcv["date"] = to_naive_date(daily_ohlcv["open_time"])
    daily_ohlcv = daily_ohlcv.groupby("date", as_index=False).agg(
        open=("open", "first"),
        high=("high", "max"),
        low=("low", "min"),
        close=("close", "last"),
        volume=("volume", "sum"),
    )

    # Execution price: open of 01:00 UTC candle
    exec_prices = klines_01[["date", "open"]].rename(columns={"open": "exec_price"})
    price_df = daily_ohlcv.merge(exec_prices, on="date", how="inner")

    print(f"Daily prices with exec price: {len(price_df)} days")
    print(f"  {price_df['date'].min().date()} to {price_df['date'].max().date()}")
    price_df.tail()

## 3. Merge Signal with Price Data

Signal timestamped day **T** uses sentiment data from day T. It becomes available at ~00:25 UTC on day **T+1** after the cron pipeline completes. Execution happens at the **open of the 01:00 UTC candle on day T+1**.

Position mapping with `shift(2)`:
- Row index aligns by **date** (= signal timestamp date).
- `exec_price` on date D is the 01:00 UTC candle open of day D.
- Signal for date T → execute at `exec_price[T+1]` → hold until `exec_price[T+2]`.
- `exec_ret = pct_change()` on row T+2 gives `exec_price[T+2] / exec_price[T+1] - 1`.
- `position.shift(2)` aligns signal T with return on row T+2 → no look-ahead bias.

In [ ]:
# Merge signal with price on date
df = signal_df.merge(price_df, on="date", how="inner").sort_values("date").reset_index(drop=True)

# Save cache (exact datetime + signal + OHLCV + exec_price, no derived columns)
if not (USE_CACHED_DATA and CACHE_FILE.exists()):
    cache_cols = ["datetime", "position", "market_indicator_human", "open", "high", "low", "close", "volume", "exec_price"]
    df[cache_cols].to_csv(CACHE_FILE, index=False)
    print(f"Cached {len(df)} rows to {CACHE_FILE}")

# Returns based on execution price (01:00 UTC candle open).
#
# Timeline for signal timestamped day T:
#   - Signal T computed from day T sentiment data
#   - Available at ~00:25 UTC on T+1
#   - Execute at 01:00 UTC on T+1 → exec_price[T+1]
#   - Hold until 01:00 UTC on T+2 → exec_price[T+2]
#   - Return = exec_price[T+2] / exec_price[T+1] - 1
#
# exec_ret (standard pct_change) on row D = exec_price[D] / exec_price[D-1] - 1
# position.shift(2) aligns signal T with exec_ret on row T+2
# → exec_price[T+2] / exec_price[T+1] - 1, which is exactly the return earned

df["exec_ret"] = df["exec_price"].pct_change()

# shift(2): signal from 2 rows back determines the position during this return period
df["active_position"] = df["position"].shift(2)

# Strategy return
df["strategy_ret"] = df["active_position"] * df["exec_ret"]

# Buy & hold return (also based on exec prices for fair comparison)
df["daily_ret"] = df["exec_ret"]

# Drop rows without valid strategy return
df = df.dropna(subset=["strategy_ret"]).copy()

# Build return series indexed by date for quantstats
strategy_returns = df.set_index("date")["strategy_ret"]
strategy_returns.index = pd.DatetimeIndex(strategy_returns.index)

benchmark_returns = df.set_index("date")["daily_ret"]
benchmark_returns.index = pd.DatetimeIndex(benchmark_returns.index)

# Position breakdown
n_long = (df["active_position"] == 1).sum()
n_neutral = (df["active_position"] == 0).sum()
n_short = (df["active_position"] == -1).sum()
total = len(df)

print(f"Merged dataset: {total} trading days")
print(f"Period: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nExecution: 01:00 UTC candle open (signal available ~00:25 UTC on T+1)")
print(f"Position shift: 2 days (signal T → exec at T+1 → return measured at T+2)")
print(f"\nPosition breakdown:")
print(f"  LONG (bullish):  {n_long:5d} days ({100*n_long/total:.1f}%)")
if n_short > 0:
    print(f"  SHORT (bearish): {n_short:5d} days ({100*n_short/total:.1f}%)")
print(f"  CASH (neutral):  {n_neutral:5d} days ({100*n_neutral/total:.1f}%)")
print(f"  Exposure:        {100*(n_long + n_short)/total:.1f}%")

## 4. Performance Summary

Key metrics comparing the sentiment strategy against buy & hold.

In [ ]:
qs.reports.metrics(strategy_returns, benchmark=benchmark_returns, mode="full")

## 5. Equity Curve & Drawdown

In [ ]:
qs.plots.returns(strategy_returns, benchmark=benchmark_returns, compound=True)

In [ ]:
qs.plots.drawdown(strategy_returns)

## 6. Monthly Returns Heatmap

In [ ]:
qs.plots.monthly_heatmap(strategy_returns)

## 7. Signal Quality Analysis

How well does the indicator separate bullish from neutral periods?

In [ ]:
# Average daily BTC return by position
long_mask = df["active_position"] == 1
neutral_mask = df["active_position"] == 0

avg_long = df.loc[long_mask, "daily_ret"].mean()
avg_neutral = df.loc[neutral_mask, "daily_ret"].mean()

print("Average daily BTC return by signal state:")
print(f"  When LONG (bullish):  {avg_long:+.4%}  (annualized: {avg_long * 365.25:+.1%})")
print(f"  When CASH (neutral):  {avg_neutral:+.4%}  (annualized: {avg_neutral * 365.25:+.1%})")
print(f"  Difference:           {avg_long - avg_neutral:+.4%}  (annualized: {(avg_long - avg_neutral) * 365.25:+.1%})")

# Signal transitions
transitions = (df["active_position"] != df["active_position"].shift(1)).sum()
avg_hold = len(df) / transitions if transitions > 0 else len(df)
print(f"\nSignal transitions: {transitions}")
print(f"Avg holding period: {avg_hold:.1f} days")

# Win rate when long
if long_mask.sum() > 0:
    long_returns = df.loc[long_mask, "daily_ret"]
    print(f"\nWin rate when LONG: {(long_returns > 0).mean():.1%}")
    print(f"Avg winning day:    {long_returns[long_returns > 0].mean():+.2%}")
    print(f"Avg losing day:     {long_returns[long_returns < 0].mean():+.2%}")

## 8. Annual Returns: Strategy vs Buy & Hold

In [ ]:
def sharpe(returns):
    """Annualized Sharpe ratio (0% risk-free rate)."""
    if returns.std() == 0:
        return 0.0
    return (returns.mean() / returns.std()) * np.sqrt(365.25)

def max_drawdown(returns):
    """Maximum drawdown from cumulative returns."""
    cum = (1 + returns).cumprod()
    return (cum / cum.cummax() - 1).min()

annual = df.copy()
annual["year"] = annual["date"].dt.year

rows = []
for year, grp in annual.groupby("year"):
    s_ret = (1 + grp["strategy_ret"]).prod() - 1
    b_ret = (1 + grp["daily_ret"]).prod() - 1
    s_sharpe = sharpe(grp["strategy_ret"])
    b_sharpe = sharpe(grp["daily_ret"])
    s_dd = max_drawdown(grp["strategy_ret"])
    b_dd = max_drawdown(grp["daily_ret"])
    long_days = (grp["active_position"] == 1).sum()
    exposure = long_days / len(grp)
    rows.append({
        "Year": year,
        "Strategy": s_ret,
        "Buy & Hold": b_ret,
        "Excess": s_ret - b_ret,
        "Sharpe (S)": s_sharpe,
        "Sharpe (B&H)": b_sharpe,
        "Sharpe Δ": s_sharpe - b_sharpe,
        "Max DD (S)": s_dd,
        "Max DD (B&H)": b_dd,
        "Max DD Δ": s_dd - b_dd,
        "Long Days": long_days,
        "Exposure": exposure,
    })

annual_df = pd.DataFrame(rows)
annual_df["Strategy"] = annual_df["Strategy"].map("{:+.1%}".format)
annual_df["Buy & Hold"] = annual_df["Buy & Hold"].map("{:+.1%}".format)
annual_df["Excess"] = annual_df["Excess"].map("{:+.1%}".format)
annual_df["Sharpe (S)"] = annual_df["Sharpe (S)"].map("{:.2f}".format)
annual_df["Sharpe (B&H)"] = annual_df["Sharpe (B&H)"].map("{:.2f}".format)
annual_df["Sharpe Δ"] = annual_df["Sharpe Δ"].map("{:+.2f}".format)
annual_df["Max DD (S)"] = annual_df["Max DD (S)"].map("{:.1%}".format)
annual_df["Max DD (B&H)"] = annual_df["Max DD (B&H)"].map("{:.1%}".format)
annual_df["Max DD Δ"] = annual_df["Max DD Δ"].map("{:+.1%}".format)
annual_df["Exposure"] = annual_df["Exposure"].map("{:.0%}".format)
annual_df

## 9. Full Quantstats Tear Sheet

Generate a comprehensive HTML report with all metrics, charts, and analysis.

In [ ]:
qs.reports.full(strategy_returns, benchmark=benchmark_returns)

## 10. Export Tear Sheet as HTML

In [ ]:
output_file = "fusion_v1_tearsheet.html"
qs.reports.html(strategy_returns, benchmark=benchmark_returns, output=output_file, title="Augmento Fusion V1 — BTC Strategy")
print(f"Tear sheet saved to {output_file}")

---

## Disclaimer

This notebook is for **informational and educational purposes only**. It does not constitute financial advice. Past performance is not indicative of future results. Always do your own research before making investment decisions.

## Links

- [Augmento](https://augmento.ai) — Crypto Sentiment Intelligence
- [API Documentation](https://augmento.ai/api-access) — Endpoints, authentication, rate limits
- [Quantstats](https://github.com/ranaroussi/quantstats) — Portfolio analytics library